# Baby Step 3 — Add Provenance, Atomic Claims, and Contradiction Control

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

This notebook adds a governed evidence layer to the five active matters while preserving Recommendation V1 and prohibiting Recommendation V2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, statistics
from collections import defaultdict

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")
if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT / "00_System" / "Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))
if 2 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 2 is not complete.")

matters = json.loads((VAULT/"data"/"active_matters.json").read_text(encoding="utf-8"))
recommendations_v1 = json.loads((VAULT/"data"/"baby_step_1_recommendations_v1.json").read_text(encoding="utf-8"))
authority_sets = json.loads((VAULT/"data"/"baby_step_1_authority_sets.json").read_text(encoding="utf-8"))
review_records = json.loads((VAULT/"data"/"baby_step_2_review_records.json").read_text(encoding="utf-8"))
precedent_list = json.loads((VAULT/"data"/"precedents.json").read_text(encoding="utf-8"))
precedents = {item["precedent_id"]: item for item in precedent_list}

print("Matters:", len(matters))
print("Recommendations V1:", len(recommendations_v1))


## Source and claim governance

Each source records origin, authority, independence, recency, directness, summary, and limitations.

Each atomic claim is classified as:

- reported fact;
- legal proposition;
- derived calculation;
- assumption;
- estimate;
- analyst inference;
- governance statement.


In [ ]:
SOURCE_TYPES = [
    "Contract","Corporate Record","Operational Record","Financial Record",
    "Synthetic Judicial Precedent","Analytical Model","Governance Decision"
]
CLAIM_TYPES = [
    "Reported Fact","Legal Proposition","Derived Calculation",
    "Assumption","Estimate","Analyst Inference","Governance Statement"
]
CONFIDENCE_WEIGHTS = {
    "authority":0.20,
    "independence":0.15,
    "recency":0.10,
    "corroboration":0.20,
    "directness":0.20,
    "consistency":0.15
}
assert abs(sum(CONFIDENCE_WEIGHTS.values())-1.0)<1e-9

(VAULT/"00_System"/"Baby_Step_3_Evidence_Governance_Model.json").write_text(
    json.dumps({
        "source_types":SOURCE_TYPES,
        "claim_types":CLAIM_TYPES,
        "confidence_weights":CONFIDENCE_WEIGHTS,
        "inference_cap":75,
        "assumption_cap":60
    },indent=2),encoding="utf-8"
)


In [ ]:
sources = []

def add_source(source_id,matter_id,source_type,origin,source_date,
               authority,independence,recency,directness,summary,limitations):
    sources.append({
        "source_id":source_id,
        "matter_id":matter_id,
        "source_type":source_type,
        "origin":origin,
        "source_date":source_date,
        "authority_score":authority,
        "independence_score":independence,
        "recency_score":recency,
        "directness_score":directness,
        "summary":summary,
        "limitations":limitations,
        "synthetic":True
    })

for matter in matters:
    mid = matter["matter_id"]
    base_sources = [
        ("Contract","Executed governing agreement",92,85,80,95,
         "Primary contractual document.","Synthetic text only."),
        ("Corporate Record","Board or committee minutes",78,70,82,85,
         "Internal governance record.","Potentially incomplete."),
        ("Operational Record","Transactional and operational records",72,75,85,80,
         "Operational evidence.","Synthetic and sample-based."),
        ("Financial Record","Damages and accounting support",76,68,88,82,
         "Financial information relevant to exposure.","Not an accounting reserve.")
    ]
    for idx,item in enumerate(base_sources,1):
        st,origin,a,i,r,d,summary,limits = item
        add_source(f"SRC-{mid}-F{idx:02d}",mid,st,origin,
                   datetime.date.today().isoformat(),a,i,r,d,summary,limits)

    candidates = authority_sets[mid]["supporting"][:3] + authority_sets[mid]["contrary"][:2]
    for idx,a in enumerate(candidates,1):
        p = precedents[a["precedent_id"]]
        authority_base = {
            "Binding":100,"Persuasive":78,"Limited Persuasive":58,
            "Unpublished":38,"Arbitral Persuasive":48
        }[p["precedential_status"]]
        penalty = {
            "Positive":0,"Followed":2,"Distinguished":20,
            "Limited":30,"Criticized":45,"Overruled":80
        }[p["treatment_status"]]
        add_source(
            f"SRC-{mid}-P{idx:02d}",mid,"Synthetic Judicial Precedent",
            p["precedent_id"],p["decision_date"],max(0,authority_base-penalty),
            90,a["components"]["recency"],a["components"]["issue_relevance"],
            p["rule_of_law"],
            f"Synthetic precedent; status={p['precedential_status']}; treatment={p['treatment_status']}."
        )

    add_source(f"SRC-{mid}-A01",mid,"Analytical Model",
               "Baby Step 1 and 2 models",datetime.date.today().isoformat(),
               55,40,100,70,"Internal synthetic analytical model.",
               "Not legal authority.")
    add_source(f"SRC-{mid}-G01",mid,"Governance Decision",
               "DEC-001 and DEC-002",datetime.date.today().isoformat(),
               85,65,100,90,"Internal governance decisions.",
               "Creates internal permission only.")

print("Sources:",len(sources))


In [ ]:
claims = []
def add_claim(claim_id,matter_id,claim_type,text,source_ids,
              dependencies,directness,corroboration,consistency,
              inference=False,assumption=False):
    claims.append({
        "claim_id":claim_id,
        "matter_id":matter_id,
        "claim_type":claim_type,
        "claim_text":text,
        "supporting_source_ids":source_ids,
        "decision_dependencies":dependencies,
        "directness_score":directness,
        "corroboration_score":corroboration,
        "consistency_score":consistency,
        "inference":inference,
        "assumption":assumption,
        "synthetic":True
    })

sources_by_matter = defaultdict(list)
for s in sources:
    sources_by_matter[s["matter_id"]].append(s)

for matter in matters:
    mid = matter["matter_id"]
    rec = next(r for r in recommendations_v1 if r["matter_id"]==mid)
    review = next(r for r in review_records if r["matter_id"]==mid)
    local = sources_by_matter[mid]
    factual = [s["source_id"] for s in local if "-F" in s["source_id"]]
    precedent_ids = [s["source_id"] for s in local if "-P" in s["source_id"]]
    analytical = [s["source_id"] for s in local if "-A" in s["source_id"]][0]
    governance = [s["source_id"] for s in local if "-G" in s["source_id"]][0]

    add_claim(f"CLM-{mid}-001",mid,"Reported Fact",
              f"The dispute concerns {matter['cause_of_action']}.",
              factual[:2],[rec["recommendation_id"]],90,80,85)
    add_claim(f"CLM-{mid}-002",mid,"Reported Fact",
              f"A principal disputed fact is: {matter['disputed_facts'][0]}.",
              factual[:3],[rec["recommendation_id"]],78,70,72)
    add_claim(f"CLM-{mid}-003",mid,"Legal Proposition",
              f"The analysis turns in part on {matter['open_questions'][0]}.",
              precedent_ids[:3],[rec["recommendation_id"]],82,75,80)
    add_claim(f"CLM-{mid}-004",mid,"Estimate",
              f"Modeled exposure is ${matter['modeled_exposure_usd']:,}.",
              [factual[-1],analytical],[rec["recommendation_id"]],70,60,68)
    add_claim(f"CLM-{mid}-005",mid,"Analyst Inference",
              f"The preferred strategy is {rec['preferred_strategy']}.",
              precedent_ids[:3]+[analytical],[rec["recommendation_id"]],
              65,62,70,inference=True)
    add_claim(f"CLM-{mid}-006",mid,"Assumption",
              f"The procedural posture remains {matter['procedural_posture']}.",
              [factual[0]],[rec["recommendation_id"]],
              75,45,70,assumption=True)
    add_claim(f"CLM-{mid}-007",mid,"Governance Statement",
              "Recommendation V1 is internal and does not authorize external action.",
              [governance],["DEC-001","DEC-002"],95,95,100)
    add_claim(f"CLM-{mid}-008",mid,"Analyst Inference",
              f"Generalization status is {review['generalization_status']}.",
              [analytical,governance],[review["review_id"]],
              72,65,82,inference=True)

print("Claims:",len(claims))


In [ ]:
source_lookup = {s["source_id"]:s for s in sources}

def confidence(claim):
    linked = [source_lookup[sid] for sid in claim["supporting_source_ids"]]
    authority = statistics.mean(s["authority_score"] for s in linked)
    independence = statistics.mean(s["independence_score"] for s in linked)
    recency = statistics.mean(s["recency_score"] for s in linked)
    raw = (
        authority*CONFIDENCE_WEIGHTS["authority"]
        + independence*CONFIDENCE_WEIGHTS["independence"]
        + recency*CONFIDENCE_WEIGHTS["recency"]
        + claim["corroboration_score"]*CONFIDENCE_WEIGHTS["corroboration"]
        + claim["directness_score"]*CONFIDENCE_WEIGHTS["directness"]
        + claim["consistency_score"]*CONFIDENCE_WEIGHTS["consistency"]
    )
    if claim["inference"]:
        raw = min(raw,75)
    if claim["assumption"]:
        raw = min(raw,60)
    return round(raw,2)

for claim in claims:
    claim["confidence_score"] = confidence(claim)

print("Average confidence:",round(statistics.mean(c["confidence_score"] for c in claims),2))


In [ ]:
contradictions = []
def add_con(cid,mid,claim_ids,category,description,decisions,severity,resolution):
    contradictions.append({
        "contradiction_id":cid,
        "matter_id":mid,
        "claim_ids":claim_ids,
        "category":category,
        "description":description,
        "affected_decisions":decisions,
        "severity":severity,
        "status":"OPEN",
        "required_resolution":resolution,
        "synthetic":True
    })

claims_by_matter = defaultdict(list)
for c in claims:
    claims_by_matter[c["matter_id"]].append(c)

for matter in matters:
    mid = matter["matter_id"]
    rec = next(r for r in recommendations_v1 if r["matter_id"]==mid)
    review = next(r for r in review_records if r["matter_id"]==mid)
    local = {c["claim_id"]:c for c in claims_by_matter[mid]}

    if review["profiled_preferred_strategy"] != rec["preferred_strategy"]:
        add_con(f"CON-{mid}-001",mid,
                [f"CLM-{mid}-005",f"CLM-{mid}-008"],
                "Strategy Conflict",
                "Recommendation V1 conflicts with the matter-specific profile.",
                [rec["recommendation_id"],review["review_id"]],
                "HIGH",
                "Human review of strategy ranking and assumptions.")

    if local[f"CLM-{mid}-004"]["confidence_score"] < 65:
        add_con(f"CON-{mid}-002",mid,[f"CLM-{mid}-004"],
                "Evidence Sufficiency",
                "Modeled exposure lacks sufficient support for widened reliance.",
                [rec["recommendation_id"]],"MEDIUM",
                "Add corroborating damages evidence.")

    if local[f"CLM-{mid}-006"]["confidence_score"] <= 60:
        add_con(f"CON-{mid}-003",mid,[f"CLM-{mid}-006"],
                "Assumption Dependency",
                "Procedural posture requires confirmation.",
                [rec["recommendation_id"]],"MEDIUM",
                "Confirm current docket posture.")

    contrary = authority_sets[mid]["contrary"]
    if contrary and any(x["treatment_status"] in ["Criticized","Overruled","Limited"] for x in contrary[:3]):
        add_con(f"CON-{mid}-004",mid,[f"CLM-{mid}-003"],
                "Authority Treatment",
                "Contrary authority may narrow the legal proposition.",
                [rec["recommendation_id"]],"HIGH",
                "Resolve treatment status and authority hierarchy.")

print("Contradictions:",len(contradictions))


In [ ]:
permission_states = []
for matter in matters:
    mid = matter["matter_id"]
    local = [c for c in contradictions if c["matter_id"]==mid]
    high = [c for c in local if c["severity"]=="HIGH"]
    medium = [c for c in local if c["severity"]=="MEDIUM"]
    permission = "HOLD_FOR_RELIANCE" if high else (
        "QUALIFIED_INTERNAL_USE" if medium else "INTERNAL_USE"
    )
    permission_states.append({
        "matter_id":mid,
        "open_contradictions":len(local),
        "high_severity":len(high),
        "medium_severity":len(medium),
        "permission_state":permission,
        "recommendation_v2_allowed":False,
        "external_action_allowed":False
    })

(VAULT/"data"/"baby_step_3_sources.json").write_text(json.dumps(sources,indent=2),encoding="utf-8")
(VAULT/"data"/"baby_step_3_claims.json").write_text(json.dumps(claims,indent=2),encoding="utf-8")
(VAULT/"data"/"baby_step_3_contradictions.json").write_text(json.dumps(contradictions,indent=2),encoding="utf-8")
(VAULT/"data"/"baby_step_3_permission_state.json").write_text(json.dumps(permission_states,indent=2),encoding="utf-8")


In [ ]:
def write_note(path,lines):
    path.write_text("\n".join(lines).strip()+"\n",encoding="utf-8")

source_dir = VAULT/"14_Sources"
claim_dir = VAULT/"15_Claims"
con_dir = VAULT/"16_Contradictions"
for d in [source_dir,claim_dir,con_dir]:
    d.mkdir(parents=True,exist_ok=True)

for s in sources:
    lines = [
        "---",f"source_id: {s['source_id']}",f"matter_id: {s['matter_id']}",
        f"source_type: \"{s['source_type']}\"","synthetic: true","---","",
        f"# {s['source_id']}","",
        f"- Matter: [[../02_Active_Matters/{s['matter_id']}]]",
        f"- Origin: {s['origin']}",
        f"- Authority: {s['authority_score']}/100",
        f"- Independence: {s['independence_score']}/100",
        f"- Recency: {s['recency_score']}/100",
        f"- Directness: {s['directness_score']}/100","",
        "## Summary","",s["summary"],"",
        "## Limitations","",s["limitations"]
    ]
    write_note(source_dir/f"{s['source_id']}.md",lines)

for c in claims:
    lines = [
        "---",f"claim_id: {c['claim_id']}",f"matter_id: {c['matter_id']}",
        f"claim_type: \"{c['claim_type']}\"",
        f"confidence_score: {c['confidence_score']}","synthetic: true","---","",
        f"# {c['claim_id']}","","## Claim","",c["claim_text"],"",
        "## Supporting sources",""
    ]
    lines += [f"- [[../14_Sources/{sid}]]" for sid in c["supporting_source_ids"]]
    lines += ["","## Decision dependencies",""]
    lines += [f"- {x}" for x in c["decision_dependencies"]]
    lines += ["","## Confidence","",f"{c['confidence_score']}/100",
              f"- Inference: {c['inference']}",f"- Assumption: {c['assumption']}"]
    write_note(claim_dir/f"{c['claim_id']}.md",lines)

for con in contradictions:
    lines = [
        "---",f"contradiction_id: {con['contradiction_id']}",
        f"matter_id: {con['matter_id']}",f"severity: {con['severity']}",
        f"status: {con['status']}","synthetic: true","---","",
        f"# {con['contradiction_id']}","",
        f"- Matter: [[../02_Active_Matters/{con['matter_id']}]]",
        f"- Category: {con['category']}",f"- Severity: {con['severity']}","",
        "## Description","",con["description"],"",
        "## Affected claims",""
    ]
    lines += [f"- [[../15_Claims/{cid}]]" for cid in con["claim_ids"]]
    lines += ["","## Required resolution","",con["required_resolution"]]
    write_note(con_dir/f"{con['contradiction_id']}.md",lines)


In [ ]:
evidence_dir = VAULT/"10_Reports"/"Evidence_Packs"
evidence_dir.mkdir(parents=True,exist_ok=True)

for matter in matters:
    mid = matter["matter_id"]
    local_claims = [c for c in claims if c["matter_id"]==mid]
    local_cons = [c for c in contradictions if c["matter_id"]==mid]
    perm = next(p for p in permission_states if p["matter_id"]==mid)
    lines = [
        f"# {mid} — Governed Evidence Pack","",
        f"- Matter: [[../../02_Active_Matters/{mid}]]",
        f"- Permission state: **{perm['permission_state']}**",
        f"- Claims: {len(local_claims)}",
        f"- Contradictions: {len(local_cons)}","",
        "## Claims by confidence",""
    ]
    for c in sorted(local_claims,key=lambda x:x["confidence_score"]):
        lines.append(
            f"- [[../../15_Claims/{c['claim_id']}]] — "
            f"{c['claim_type']} — {c['confidence_score']}/100"
        )
    lines += ["","## Open contradictions",""]
    lines += [
        f"- [[../../16_Contradictions/{c['contradiction_id']}]] — "
        f"{c['severity']} — {c['category']}"
        for c in local_cons
    ] or ["- None"]
    write_note(evidence_dir/f"{mid}_Evidence_Pack.md",lines)

report = [
    "# Baby Step 3 — Provenance and Contradiction Report","",
    f"- Sources: **{len(sources)}**",
    f"- Atomic claims: **{len(claims)}**",
    f"- Contradictions: **{len(contradictions)}**",
    f"- Average claim confidence: **{statistics.mean(c['confidence_score'] for c in claims):.2f}/100**","",
    "## Matter permission states",""
]
report += [
    f"- {p['matter_id']}: **{p['permission_state']}** — "
    f"{p['open_contradictions']} open contradictions"
    for p in permission_states
]
report += ["","Recommendation V1 remains preserved. No Recommendation V2 is authorized."]
write_note(VAULT/"10_Reports"/"Baby_Step_3_Provenance_Contradiction_Report.md",report)


In [ ]:
DECISION = {
    "decision_id":"DEC-003",
    "date":datetime.date.today().isoformat(),
    "title":"Accept Provenance and Contradiction Controls",
    "decision":"Accept the source, claim, confidence, contradiction, and permission architecture as the governed evidence baseline.",
    "permitted_next_actions":[
        "internal evidence clarification",
        "contradiction resolution",
        "source-quality improvement",
        "committee-product preparation using qualified claims"
    ],
    "not_authorized":[
        "Recommendation V2",
        "reliance on high-severity contradicted claims",
        "filing","service","party contact","court contact",
        "external counsel instruction","settlement offer",
        "external legal advice","external distribution"
    ],
    "synthetic":True
}
(VAULT/"09_Decisions"/"DEC-003.json").write_text(json.dumps(DECISION,indent=2),encoding="utf-8")
lines = [
    "# DEC-003 — Accept Provenance and Contradiction Controls","",
    f"**Date:** {DECISION['date']}","","## Decision","",DECISION["decision"],"",
    "## Permitted next actions",""
]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]
write_note(VAULT/"09_Decisions"/"DEC-003.md",lines)


In [ ]:
hot = [
    "# Current State — Hot Cache","",
    "## Recommendation state","",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.","",
    "## Evidence-governance state","",
    f"- Sources: {len(sources)}",
    f"- Atomic claims: {len(claims)}",
    f"- Open contradictions: {len(contradictions)}","",
    "## Matter permission states",""
]
hot += [f"- {p['matter_id']}: **{p['permission_state']}**" for p in permission_states]
hot += [
    "","## Current decision","",
    "- [[../09_Decisions/DEC-003]]","",
    "## Next permitted experiment","",
    "Produce the first committee-ready five-matter litigation product."
]
write_note(VAULT/"12_Hot_Cache"/"Current_State.md",hot)


In [ ]:
errors = []
source_notes = list((VAULT/"14_Sources").glob("*.md"))
claim_notes = list((VAULT/"15_Claims").glob("*.md"))
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))

if len(v1)!=5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")
if v2:
    errors.append("Recommendation V2 exists prematurely")
if not source_notes:
    errors.append("No source notes")
if not claim_notes:
    errors.append("No claim notes")
for claim in claims:
    if not claim["supporting_source_ids"]:
        errors.append(f"{claim['claim_id']}: no source")
    if not (0<=claim["confidence_score"]<=100):
        errors.append(f"{claim['claim_id']}: invalid confidence")
if len(permission_states)!=5:
    errors.append("Missing permission states")

required = [
    VAULT/"09_Decisions"/"DEC-003.md",
    VAULT/"10_Reports"/"Baby_Step_3_Provenance_Contradiction_Report.md",
    VAULT/"data"/"baby_step_3_sources.json",
    VAULT/"data"/"baby_step_3_claims.json",
    VAULT/"data"/"baby_step_3_contradictions.json",
    VAULT/"data"/"baby_step_3_permission_state.json"
]
for p in required:
    if not p.exists():
        errors.append(f"Missing: {p}")

validation = {
    "validated_at":datetime.datetime.now().isoformat(),
    "source_note_count":len(source_notes),
    "claim_note_count":len(claim_notes),
    "contradiction_count":len(contradictions),
    "recommendation_v1_count":len(v1),
    "recommendation_v2_count":len(v2),
    "decision":"DEC-003",
    "errors":errors,
    "passed":len(errors)==0
}
(VAULT/"11_Audit"/"Baby_Step_3_Validation.json").write_text(
    json.dumps(validation,indent=2),encoding="utf-8"
)
assert validation["passed"],errors
print(json.dumps(validation,indent=2))
print("BABY STEP 3 PASSED")


In [ ]:
state.update({
    "completed_steps":sorted(set(state.get("completed_steps",[])+[3])),
    "current_step":3,
    "next_step":4,
    "decision":"DEC-003",
    "source_count":len(sources),
    "claim_count":len(claims),
    "contradiction_count":len(contradictions),
    "next_problem":"Produce the first committee-ready five-matter litigation product.",
    "permission_state":{
        "observe":True,
        "organize":True,
        "browse":True,
        "internal_strategy_analysis":True,
        "evidence_governance":True,
        "committee_product_preparation":True,
        "recommendation_v2":False,
        "external_action":False
    }
})
state_path.write_text(json.dumps(state,indent=2),encoding="utf-8")
audit = {
    "timestamp":datetime.datetime.now().isoformat(),
    "step":3,
    "action":"Added provenance, atomic claims, confidence, contradictions, and permission controls.",
    "outputs":{"sources":len(sources),"claims":len(claims),
               "contradictions":len(contradictions),"decision":"DEC-003"},
    "validation_passed":True
}
with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a",encoding="utf-8") as f:
    f.write(json.dumps(audit)+"\n")
